# Laboratoire 9 : VQE et chimie quantique

**Semaine 11 — Cours de Quantum Machine Learning (Master/PhD)**

## Objectifs

- Construire l'Hamiltonien moléculaire de H₂ (et LiH) avec PennyLane `qml.qchem`
- Implémenter l'ansatz UCCSD (Unitary Coupled Cluster Singles Doubles)
- Optimiser l'énergie de l'état fondamental par VQE avec Adam et SPSA
- Comparer avec la diagonalisation exacte (`numpy.linalg.eigh`)
- Utiliser Qiskit Nature avec PySCFDriver pour le VQE
- Tracer la courbe de dissociation de H₂ et analyser le gap d'énergie

---
## Imports

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Configuration du tracé
plt.rcParams.update({'font.size': 12, 'figure.figsize': (8, 5)})

print(f"PennyLane version : {qml.__version__}")

---
## Partie A : Construction de l'Hamiltonien moléculaire

Nous allons construire l'Hamiltonien électronique de la molécule H₂ dans une base minimale STO-3G à l'aide de `qml.qchem`. L'Hamiltonien est exprimé sous forme de combinaison linéaire d'opérateurs de Pauli agissant sur des qubits (encodage de Jordan-Wigner).

In [ ]:
# Paramètres de la molécule H₂
symbols = ["H", "H"]
bond_length = 0.74  # Å (distance d'équilibre)
geometry = np.array([[0.0, 0.0, 0.0], [bond_length, 0.0, 0.0]])

# Construction de l'Hamiltonien via PennyLane qchem
H, qubits = qml.qchem.molecular_hamiltonian(
    symbols,
    geometry,
    charge=0,
    basis="sto-3g"
)

print(f"Nombre de qubits : {qubits}")
print(f"Nombre de termes dans l'Hamiltonien : {len(H.ops)}")
print(f"\nHamiltonien :\n{H}")

### 1.1 Diagonalisation exacte

Pour référence, diagonalisons la matrice Hamiltonienne complète pour obtenir l'énergie exacte de l'état fondamental.

In [ ]:
# Représentation matricielle de l'Hamiltonien
H_mat = qml.matrix(H)
eigvals, eigvecs = np.linalg.eigh(H_mat)
exact_gs_energy = eigvals[0]

print(f"Énergie exacte de l'état fondamental : {exact_gs_energy:.8f} Ha")
print(f"Spectre complet : {eigvals}")

---
## Partie B : Ansatz UCCSD et Hartree-Fock

Le ansatz UCCSD (Unitary Coupled Cluster Singles Doubles) est l'un des plus utilisés en chimie quantique variationnelle. Il construit un état trial à partir du déterminant de Hartree-Fock en appliquant des excitations simples et doubles.

In [ ]:
# Configuration de l'ansatz
electrons = 2  # H₂ a 2 électrons

# Ansatz UCCSD avec PennyLane
ansatz = qml.qchem.UCCSD

# Récupération des paramètres initiaux (HF state) et des excitations
hf_state = qml.qchem.hf_state(electrons, qubits)
print(f"État Hartree-Fock : {hf_state}")

# Génération des excitations
singles, doubles = qml.qchem.excitations(electrons, qubits)
print(f"Excitation simple : {singles}")
print(f"Excitation double : {doubles}")

n_params = len(singles) + len(doubles)
print(f"Nombre total de paramètres variationnels : {n_params}")

### Définition du circuit VQE

In [ ]:
dev = qml.device("default.qubit", wires=qubits)

@qml.qnode(dev, diff_method="parameter-shift")
def cost_fn(params):
    ansatz(params, wires=range(qubits), hf_state=hf_state, singles=singles, doubles=doubles)
    return qml.expval(H)

---
## Partie C : Optimisation VQE

Nous testons deux optimiseurs : **Adam** (basé sur le gradient) et **SPSA** (sans gradient explicite, adapté au bruit).

In [ ]:
# Paramètres d'optimisation
np.random.seed(42)
init_params = np.random.uniform(-0.5, 0.5, size=n_params)
n_epochs = 200

def optimize_vqe(opt_class, opt_kwargs, init_params, n_epochs, label):
    params = init_params.copy()
    opt = opt_class(**opt_kwargs)
    energies = []
    
    for i in range(n_epochs):
        params, energy = opt.step_and_cost(cost_fn, params)
        energies.append(energy)
        if (i + 1) % 50 == 0:
            print(f"[{label}] Étape {i+1:3d} : énergie = {energy:.6f} Ha")
    
    return params, energies

# Optimisation avec Adam
print("=== Optimisation Adam ===")
params_adam, energies_adam = optimize_vqe(
    qml.AdamOptimizer, {"stepsize": 0.1}, init_params, n_epochs, "Adam"
)

# Optimisation avec SPSA
print("\n=== Optimisation SPSA ===")
params_spsa, energies_spsa = optimize_vqe(
    qml.SPSAOptimizer, {"stepsize": 0.1}, init_params, n_epochs, "SPSA"
)

In [ ]:
# Comparaison des résultats
print(f"Énergie exacte (diagonalisation) : {exact_gs_energy:.8f} Ha")
print(f"Adam  - Énergie finale : {energies_adam[-1]:.8f} Ha, erreur : {abs(energies_adam[-1] - exact_gs_energy):.2e} Ha")
print(f"SPSA  - Énergie finale : {energies_spsa[-1]:.8f} Ha, erreur : {abs(energies_spsa[-1] - exact_gs_energy):.2e} Ha")

# Tracé des courbes de convergence
plt.figure()
plt.plot(energies_adam, label="Adam", lw=2)
plt.plot(energies_spsa, label="SPSA", lw=2)
plt.axhline(exact_gs_energy, color="black", ls="--", label=f"Exacte ({exact_gs_energy:.4f} Ha)")
plt.xlabel("Itération")
plt.ylabel("Énergie [Ha]")
plt.title("Convergence VQE : Adam vs SPSA")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Partie D : Qiskit Nature — PySCFDriver et VQE avec Estimator

Nous utilisons maintenant Qiskit Nature pour construire l'Hamiltonien via PySCF et exécuter le VQE avec l'Estimator primitif.

In [ ]:
try:
    from qiskit_nature.units import DistanceUnit
    from qiskit_nature.second_q.drivers import PySCFDriver
    from qiskit_nature.second_q.mappers import ParityMapper, JordanWignerMapper
    from qiskit_nature.second_q.algorithms import GroundStateEigensolver
    from qiskit_nature.second_q.circuit.library import UCCSD
    from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
    
    from qiskit_algorithms import VQE
    from qiskit_algorithms.optimizers import SLSQP
    from qiskit.primitives import Estimator
    from qiskit.opflow import PauliSumOp
    
    HAS_QISKIT_NATURE = True
    print("Qiskit Nature disponible.")
except ImportError as e:
    HAS_QISKIT_NATURE = False
    print(f"Qiskit Nature non disponible : {e}")
    print("Installez avec : pip install qiskit-nature pyscf")

In [ ]:
if HAS_QISKIT_NATURE:
    # Driver PySCF pour H₂
    driver = PySCFDriver(
        atom=f"H 0 0 0; H 0 0 {bond_length}",
        basis="sto-3g",
        charge=0,
        spin=0,
        unit=DistanceUnit.ANGSTROM
    )
    
    # Propriétés électroniques
    problem = driver.run()
    print(f"Hamiltonien généré par PySCF : {problem.num_spatial_orbitals} orbitales spatiales")
    
    # Mapper de Jordan-Wigner
    mapper = JordanWignerMapper()
    
    # Ansatz UCCSD
    ansatz_qiskit = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
        initial_state=None
    )
    
    # VQE avec Estimator
    estimator = Estimator()
    optimizer = SLSQP(maxiter=200)
    
    vqe = VQE(estimator, ansatz_qiskit, optimizer)
    
    # Algorithme du solveur d'état fondamental
    solver = GroundStateEigensolver(mapper, vqe)
    
    # Résultat
    result = solver.solve(problem)
    print(f"\nÉnergie VQE (Qiskit Nature) : {result.total_energies[0]:.8f} Ha")
    print(f"Énergie exacte (référence)     : {exact_gs_energy:.8f} Ha")
    print(f"Erreur : {abs(result.total_energies[0] - exact_gs_energy):.2e} Ha")

---
## Partie E : Courbe de dissociation de H₂

Nous faisons varier la distance internucléaire pour tracer la courbe d'énergie potentielle et calculer le gap HOMO-LUMO.

In [ ]:
bond_lengths = np.linspace(0.4, 3.0, 15)
energies_vqe = []
energies_exact = []

for d in bond_lengths:
    geo = np.array([[0.0, 0.0, 0.0], [d, 0.0, 0.0]])
    H_d, _ = qml.qchem.molecular_hamiltonian(symbols, geo, charge=0, basis="sto-3g")
    
    # Énergie exacte
    H_mat_d = qml.matrix(H_d)
    eigs_d = np.linalg.eigh(H_mat_d)[0]
    energies_exact.append(eigs_d[0])
    
    # VQE avec Adam
    dev_d = qml.device("default.qubit", wires=qubits)
    
    @qml.qnode(dev_d, diff_method="parameter-shift")
    def cost_d(params):
        ansatz(params, wires=range(qubits), hf_state=hf_state, singles=singles, doubles=doubles)
        return qml.expval(H_d)
    
    params_d = np.random.uniform(-0.5, 0.5, size=n_params)
    opt_d = qml.AdamOptimizer(stepsize=0.1)
    for _ in range(100):
        params_d, _ = opt_d.step_and_cost(cost_d, params_d)
    
    energies_vqe.append(cost_d(params_d))

energies_vqe = np.array(energies_vqe)
energies_exact = np.array(energies_exact)

In [ ]:
# Tracé de la courbe de dissociation
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(bond_lengths, energies_exact, "o-", label="Exact (diagonalisation)", lw=2)
plt.plot(bond_lengths, energies_vqe, "s--", label="VQE (Adam)", lw=2)
plt.xlabel("Distance internucléaire [Å]")
plt.ylabel("Énergie [Ha]")
plt.title("Courbe de dissociation H₂")
plt.legend()
plt.grid(True, alpha=0.3)

# Erreur
plt.subplot(1, 2, 2)
plt.semilogy(bond_lengths, np.abs(energies_vqe - energies_exact), "ro-", lw=2)
plt.xlabel("Distance internucléaire [Å]")
plt.ylabel("Erreur absolue [Ha]")
plt.title("Erreur VQE vs exacte")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calcul du gap HOMO-LUMO à l'équilibre
gap = eigvals[1] - eigvals[0]
print(f"\nGap HOMO-LUMO à d = {bond_length:.2f} Å : {gap:.6f} Ha = {gap * 27.2114:.3f} eV")

---
## Extension à LiH

Pour explorer une molécule plus grande, construisons l'Hamiltonien de LiH (4 électrons, 6 qubits après encodage).

In [ ]:
# Paramètres LiH
symbols_lih = ["Li", "H"]
bond_length_lih = 1.595  # Å
geometry_lih = np.array([[0.0, 0.0, 0.0], [bond_length_lih, 0.0, 0.0]])

# Hamiltonien LiH
H_lih, qubits_lih = qml.qchem.molecular_hamiltonian(
    symbols_lih,
    geometry_lih,
    charge=0,
    basis="sto-3g"
)

print(f"Nombre de qubits pour LiH : {qubits_lih}")
print(f"Nombre de termes : {len(H_lih.ops)}")

# Énergie exacte
H_lih_mat = qml.matrix(H_lih)
e_lih = np.linalg.eigh(H_lih_mat)[0][0]
print(f"Énergie fondamentale exacte LiH : {e_lih:.8f} Ha")

---
## Exercices — Pour aller plus loin

1. **Molécule H₂O** : Construisez l'Hamiltonien de H₂O (3 atomes) en base STO-3G. Combien de qubits ?
2. **Ansatz comparé** : Implémentez un ansatz hardware-efficient simple (couches de RX + CNOT) et comparez la convergence avec UCCSD.
3. **Optimiseurs** : Testez SLSQP, L-BFGS-B, COBYLA. Comparez le nombre d'itérations et l'énergie finale.
4. **Molécule à l'état excité** : Utilisez `qml.qchem` pour calculer le premier état excité de H₂ en modifiant la fonction de coût (folded spectrum ou penalty).
5. **Bruit** : Ajoutez un modèle de bruit déphasant avec `default.mixed` ou Qiskit Aer. Comment évolue l'erreur VQE ?
6. **Analyse de courbe** : La courbe de dissociation montre-t-elle un changement de pente ? Que se passe-t-il pour H₂ au-delà de 2.5 Å ?